## 算法目的说明

这个算法的目的是为论文 *Mixed motives and linear forms in the Catalan constant* 中 Remark 8.3.4 提供一个可计算的验证框架。该 Remark 表明：若能在菱形区域

$$
S=\{(x,y)\in\mathbb R^2:\ |x|+|y|\le 1\}
$$

上找到一个非零整系数多项式 $P(x,y)\in\mathbb Z[x,y]$，满足

$$
\|P\|_S^{1/\deg P}<\frac{1}{2e^{3/2}},
$$

则可以通过构造 $G_j=P^j$ 以及

$$
F_j=\sum_{i=0}^3 \sigma^i(G_j^2),\qquad \sigma(x,y)=(-y,x),
$$

得到论文所需的一列足够小的线性形式 $a_j+b_jG$，从而推出 Catalan 常数 $G=\beta(2)$ 的无理性。因此，原本抽象的“整数 Chebyshev 常数是否足够小”的问题，可以转化为一个具体的有限计算问题：在给定次数 $d$ 下，寻找整系数多项式 $P$，使得

$$
\sup_{|x|+|y|\le1}|P(x,y)| < (2e^{3/2})^{-d}.
$$

算法采用 Bernstein 多项式方法来严格上界估计 $\|P\|_S$。具体地，将菱形区域 $S$ 分解为四个三角形，并在每个三角形上把 $P$ 展开为 Bernstein 基形式。由于 Bernstein 基函数在三角形上非负且和为 $1$，多项式在该三角形上的取值被其 Bernstein 系数的最大绝对值控制。因此，只要四个三角形上的所有 Bernstein 系数绝对值都小于 $(2e^{3/2})^{-d}$，就得到一个严格的、可验证的证书。

该算法分为两部分：第一部分是验证器，用精确有理数计算给定多项式 $P$ 的 Bernstein 上界；第二部分是候选搜索器，把 Bernstein 系数视为整系数 $c_{ij}$ 的有理线性函数，并用整数线性规划或格约化方法尝试寻找使这些系数同时很小的非零整系数多项式。若搜索成功并通过 Bernstein 验证，则不仅得到一个数值现象，而是得到一个严格可检查的无理性证明证书；若长期搜索失败，也可以作为该 Chebyshev 路线可能难以奏效的计算证据。

comment：我的意见，这个多项式大概率是不存在的。。。。

In [2]:
# SageMath code

from sage.all import *

# Polynomial ring
R = PolynomialRing(QQ, "x,y")
x, y = R.gens()


def falling_factorial_QQ(n, k):
    """
    Return n*(n-1)*...*(n-k+1) as QQ.
    """
    if k < 0:
        return QQ(0)
    if k == 0:
        return QQ(1)
    if n < k:
        return QQ(0)
    prod = QQ(1)
    for r in range(k):
        prod *= QQ(n - r)
    return prod


def bernstein_coeffs_standard_triangle(P, d=None):
    """
    Compute Bernstein coefficients of P on the standard triangle

        Delta = {(x,y): x >= 0, y >= 0, x+y <= 1}

    using total Bernstein degree d.

    Coordinates:
        lambda_1 = x
        lambda_2 = y
        lambda_3 = 1 - x - y

    Bernstein basis:
        B_{a,b,c}^{(d)} = d!/(a! b! c!) * lambda_1^a lambda_2^b lambda_3^c

    Returns:
        dict mapping (a,b,c) with a+b+c=d to the Bernstein coefficient.
    """
    P = R(P)
    if d is None:
        d = P.total_degree()

    if d < P.total_degree():
        raise ValueError("Bernstein degree d must be at least total degree of P.")

    monoms = P.monomial_coefficients()  # keys are exponent tuples (i,j), values are coefficients

    coeffs = {}

    for a in range(d + 1):
        for b in range(d + 1 - a):
            c = d - a - b

            Babc = QQ(0)

            for (i, j), cij in monoms.items():
                if i + j > d:
                    continue

                # coefficient of x^i y^j in Bernstein degree d
                # at index (a,b,c):
                #
                # (a)_i (b)_j / (d)_{i+j}
                #
                # with the convention that it is 0 if a<i or b<j.
                numerator = falling_factorial_QQ(a, i) * falling_factorial_QQ(b, j)
                denominator = falling_factorial_QQ(d, i + j)

                if denominator != 0:
                    Babc += QQ(cij) * numerator / denominator

            coeffs[(a, b, c)] = Babc

    return coeffs


def rotate_polynomial(P, k):
    """
    Apply sigma^k to the variables, where

        sigma(x,y) = (-y, x).

    For checking P on sigma^k(Delta), it suffices to check
    P(sigma^k(x,y)) on Delta.
    """
    P = R(P)
    k = k % 4

    if k == 0:
        return R(P)
    elif k == 1:
        return R(P(-y, x))
    elif k == 2:
        return R(P(-x, -y))
    elif k == 3:
        return R(P(y, -x))


def bernstein_bound_on_diamond(P, d=None, return_details=False):
    """
    Compute a rigorous Bernstein upper bound for

        sup_{|x|+|y| <= 1} |P(x,y)|.

    The diamond is decomposed as

        S = Delta union sigma(Delta) union sigma^2(Delta) union sigma^3(Delta).

    Returns:
        B = max absolute Bernstein coefficient over all four triangles.

    This is a rigorous upper bound:
        ||P||_S <= B.
    """
    P = R(P)
    if d is None:
        d = P.total_degree()

    global_bound = QQ(0)
    details = []

    for k in range(4):
        Q = rotate_polynomial(P, k)
        coeffs = bernstein_coeffs_standard_triangle(Q, d=d)

        local_bound = max(abs(v) for v in coeffs.values())
        details.append((k, Q, local_bound, coeffs))

        if local_bound > global_bound:
            global_bound = local_bound

    if return_details:
        return global_bound, details
    else:
        return global_bound


def chebyshev_threshold(d, prec=200):
    """
    Return interval approximation to (2 e^(3/2))^(-d).
    """
    RIF = RealIntervalField(prec)
    e = RIF(1).exp()
    return (RIF(2) * e**(RIF(3)/RIF(2)))**(-d)


def verify_catalan_certificate(P, d=None, prec=200, verbose=True):
    """
    Verify whether the Bernstein upper bound proves

        ||P||_S < (2 e^(3/2))^(-d).

    Since the RHS is transcendental, comparison is done using interval arithmetic.

    If Bernstein_bound < lower_bound(threshold interval), then the verification is rigorous.
    """
    P = R(P)

    if d is None:
        d = P.total_degree()

    if P == 0:
        raise ValueError("P must be nonzero.")

    B = bernstein_bound_on_diamond(P, d=d)
    T = chebyshev_threshold(d, prec=prec)

    B_interval = RealIntervalField(prec)(B)

    success = B_interval.upper() < T.lower()

    if verbose:
        print("P =", P)
        print("degree d =", d)
        print("Bernstein upper bound B =", B)
        print("threshold T = (2 e^(3/2))^(-d) in interval form:")
        print(T)
        print("B < T ?", success)

        if success:
            print("SUCCESS: This P would satisfy the sufficient Chebyshev certificate.")
        else:
            print("FAIL: Bernstein bound is not small enough.")

    return success, B, T

In [3]:
# === Enhanced verification: degree elevation + subdivision + true sup-norm ===

DIAMOND_TRIANGLES = [
    ((QQ(0),QQ(0)), (QQ(1),QQ(0)), (QQ(0),QQ(1))),
    ((QQ(0),QQ(0)), (QQ(0),QQ(1)), (QQ(-1),QQ(0))),
    ((QQ(0),QQ(0)), (QQ(-1),QQ(0)), (QQ(0),QQ(-1))),
    ((QQ(0),QQ(0)), (QQ(0),QQ(-1)), (QQ(1),QQ(0))),
]


def bernstein_bound_on_subtriangle(P, v0, v1, v2, bern_deg=None):
    """
    Bernstein bound of |P| on triangle with vertices v0, v1, v2.
    Affine substitution maps standard triangle (x,y) -> sub-triangle.
    """
    X = QQ(v0[0])*x + QQ(v1[0])*y + QQ(v2[0])*(1 - x - y)
    Y = QQ(v0[1])*x + QQ(v1[1])*y + QQ(v2[1])*(1 - x - y)
    Q = R(P(X, Y))
    if bern_deg is None:
        bern_deg = Q.total_degree()
    coeffs = bernstein_coeffs_standard_triangle(Q, d=bern_deg)
    return max(abs(v) for v in coeffs.values())


def subdivide_triangle(v0, v1, v2):
    """Midpoint subdivision into 4 sub-triangles."""
    m01 = ((v0[0]+v1[0])/2, (v0[1]+v1[1])/2)
    m02 = ((v0[0]+v2[0])/2, (v0[1]+v2[1])/2)
    m12 = ((v1[0]+v2[0])/2, (v1[1]+v2[1])/2)
    return [(v0, m01, m02), (v1, m12, m01), (v2, m02, m12), (m01, m12, m02)]


def bernstein_bound_tight(P, bern_deg=None, subdiv_depth=0):
    """
    Rigorous upper bound on ||P||_S using degree elevation + subdivision.

    bern_deg: Bernstein expansion degree (higher = tighter). Default: 2*deg(P).
    subdiv_depth: recursive midpoint subdivisions. Total sub-triangles = 4^(depth+1).
    """
    P = R(P)
    if bern_deg is None:
        bern_deg = 2 * P.total_degree()

    triangles = list(DIAMOND_TRIANGLES)
    for _ in range(subdiv_depth):
        new_tri = []
        for tri in triangles:
            new_tri.extend(subdivide_triangle(*tri))
        triangles = new_tri

    bound = QQ(0)
    for v0, v1, v2 in triangles:
        b = bernstein_bound_on_subtriangle(P, v0, v1, v2, bern_deg=bern_deg)
        if b > bound:
            bound = b
    return bound


def numerical_sup_on_diamond(P, grid_size=300):
    """
    Evaluate |P| on a grid over S to estimate true ||P||_S (lower bound).
    """
    P = R(P)
    max_val = QQ(0)
    for i in range(grid_size + 1):
        for j in range(grid_size + 1 - i):
            xi = QQ(i) / grid_size
            yi = QQ(j) / grid_size
            for sx, sy in [(1,1), (-1,1), (-1,-1), (1,-1)]:
                val = abs(P(sx*xi, sy*yi))
                if val > max_val:
                    max_val = val
    return max_val


def verify_catalan_v2(P, d=None, bern_deg=None, subdiv_depth=2,
                      grid_size=300, prec=200, verbose=True):
    """
    Enhanced verification with degree elevation, subdivision, and numerical estimate.
    """
    P = R(P)
    if d is None:
        d = P.total_degree()
    if P == 0:
        raise ValueError("P must be nonzero.")
    if bern_deg is None:
        bern_deg = 2 * d

    B_naive = bernstein_bound_on_diamond(P, d=d)
    B_elevated = bernstein_bound_tight(P, bern_deg=bern_deg, subdiv_depth=0)
    B_subdiv = bernstein_bound_tight(P, bern_deg=bern_deg, subdiv_depth=subdiv_depth)
    sup_num = numerical_sup_on_diamond(P, grid_size=grid_size)

    T = chebyshev_threshold(d, prec=prec)
    RIF = RealIntervalField(prec)
    success = RIF(B_subdiv).upper() < T.lower()

    if verbose:
        n_tri = 4 * 4**subdiv_depth
        print("P =", P)
        print("degree d =", d)
        print()
        print("--- Bounds comparison ---")
        print("  Numerical ||P||_S  >=  %s" % RR(sup_num))
        print("  Bernstein naive (D=d=%d):  %s" % (d, B_naive))
        print("  Degree elevation (D=%d):   %s  (%.6f)" % (bern_deg, B_elevated, RR(B_elevated)))
        print("  + Subdivision (depth=%d, %d triangles):  %s  (%.6f)" % (
            subdiv_depth, n_tri, B_subdiv, RR(B_subdiv)))
        print()
        print("  Threshold T = (2e^(3/2))^(-%d) ~ %.6e" % (d, RR(T.center())))
        print("  B_best / T ~ %.2f" % (RR(B_subdiv) / RR(T.center())))
        print()
        if success:
            print("SUCCESS: Rigorous certificate verified.")
        else:
            print("FAIL: B_best < T not satisfied.")
            gap = RR(B_subdiv) / RR(sup_num)
            print("  Bound / true sup ~ %.4f (tightness ratio, 1.0 = perfect)" % gap)

    return success, B_subdiv, T

In [4]:
P = 1 - x - y
verify_catalan_certificate(P)

P = -x - y + 1
degree d = 1
Bernstein upper bound B = 2
threshold T = (2 e^(3/2))^(-d) in interval form:
0.111565080074214914466640235382006260671085814680539664371918?
B < T ? False
FAIL: Bernstein bound is not small enough.


(False, 2, 0.111565080074214914466640235382006260671085814680539664371918?)

### 结果解读：`P = 1 - x - y`

**为什么 FAIL？**

验证条件要求 Bernstein 上界 $B$ 严格小于 Chebyshev 阈值 $T = (2e^{3/2})^{-d}$。对 $P = 1 - x - y$（$d = 1$）：

- **Bernstein 上界 $B = 2$**：菱形 $S = \{|x|+|y| \le 1\}$ 被分解为 4 个标准三角形（通过旋转 $\sigma: (x,y)\mapsto(-y,x)$）。在 $k=2$ 旋转下 $P(-x,-y) = 1+x+y$，其 Bernstein 系数为 $b_{(1,0,0)} = 2,\; b_{(0,1,0)} = 2,\; b_{(0,0,1)} = 1$，最大绝对值为 $2$。
- **阈值 $T \approx 0.1116$**：$(2e^{3/2})^{-1}$ 远小于 $1$。

$B = 2 \gg T \approx 0.1116$，条件 $B < T$ 不成立。

**直觉**：$P$ 在菱形顶点 $(-1,0)$ 处取值 $P(-1,0) = 2$，即 $\|P\|_S \ge 2$。而 Chebyshev 阈值对 $d=1$ 要求 sup-norm $< 0.11$，任何非平凡线性多项式都无法满足。

In [5]:
P = 1 - 6*x**2 - 6*y**2 + 20*x**2*y**2
verify_catalan_certificate(P)

P = 20*x^2*y^2 - 6*x^2 - 6*y^2 + 1
degree d = 4
Bernstein upper bound B = 5
threshold T = (2 e^(3/2))^(-d) in interval form:
0.00015492201104164740144032296442604174321915497409587156567992?
B < T ? False
FAIL: Bernstein bound is not small enough.


(False, 5, 0.00015492201104164740144032296442604174321915497409587156567992?)

In [6]:
P = 1 - 6*x**2 - 6*y**2 + 20*x**2*y**2

B, details = bernstein_bound_on_diamond(P, return_details=True)

print("Global Bernstein bound:", B)

for k, Q, local_bound, coeffs in details:
    print()
    print("Triangle sigma^%s(Delta)" % k)
    print("Pulled-back polynomial:", Q)
    print("Local Bernstein bound:", local_bound)

Global Bernstein bound: 5

Triangle sigma^0(Delta)
Pulled-back polynomial: 20*x^2*y^2 - 6*x^2 - 6*y^2 + 1
Local Bernstein bound: 5

Triangle sigma^1(Delta)
Pulled-back polynomial: 20*x^2*y^2 - 6*x^2 - 6*y^2 + 1
Local Bernstein bound: 5

Triangle sigma^2(Delta)
Pulled-back polynomial: 20*x^2*y^2 - 6*x^2 - 6*y^2 + 1
Local Bernstein bound: 5

Triangle sigma^3(Delta)
Pulled-back polynomial: 20*x^2*y^2 - 6*x^2 - 6*y^2 + 1
Local Bernstein bound: 5


In [7]:
def monomial_basis_total_degree(d):
    """
    Return list of exponent pairs (i,j) with i+j <= d.
    Ordered by total degree, then i.
    """
    basis = []
    for total in range(d + 1):
        for i in range(total + 1):
            j = total - i
            basis.append((i, j))
    return basis


def polynomial_from_coeff_vector(coeffs, basis):
    """
    Build polynomial sum coeffs[k] * x^i y^j.
    """
    P = R(0)
    for ck, (i, j) in zip(coeffs, basis):
        P += ZZ(ck) * x**i * y**j
    return R(P)


def bernstein_linear_matrix_for_diamond(d):
    """
    Build the rational matrix A such that

        A * c

    is the list of Bernstein coefficients of P on all four triangles,
    where c is the coefficient vector of P in the total-degree monomial basis.

    Rows correspond to:
        k = 0,1,2,3 and (a,b,c) with a+b+c=d.

    Columns correspond to monomials x^i y^j with i+j <= d.

    Returns:
        A, basis, row_labels
    """
    basis = monomial_basis_total_degree(d)
    rows = []
    row_labels = []

    monomial_polys = [x**i * y**j for (i, j) in basis]

    for k in range(4):
        transformed_monomials = [rotate_polynomial(M, k) for M in monomial_polys]

        # For each transformed monomial, compute its Bernstein coefficients.
        transformed_coeffs = [
            bernstein_coeffs_standard_triangle(Mk, d=d)
            for Mk in transformed_monomials
        ]

        for a in range(d + 1):
            for b in range(d + 1 - a):
                c = d - a - b

                row = []
                for col in range(len(basis)):
                    row.append(transformed_coeffs[col][(a, b, c)])

                rows.append(row)
                row_labels.append((k, a, b, c))

    A = matrix(QQ, rows)
    return A, basis, row_labels


def search_candidate_by_milp(d, coeff_bound=10, pivot_index=None, pivot_sign=1, verbose=True):
    """
    Heuristic bounded search using Mixed Integer Linear Programming.

    It tries to minimize R subject to

        -R <= BernsteinCoeff_i(P) <= R

    for all Bernstein coefficients on all four triangles.

    To avoid the zero polynomial, we fix one coefficient:
        c[pivot_index] = pivot_sign.

    Parameters:
        d: total degree
        coeff_bound: impose -coeff_bound <= c_j <= coeff_bound
        pivot_index: which monomial coefficient to force to +1 or -1.
                     If None, use the highest-degree last basis vector.
        pivot_sign: +1 or -1

    Returns:
        P, objective_value

    Warning:
        This uses a numerical MILP solver, usually GLPK.
        Always verify the output with verify_catalan_certificate(P).
    """
    A, basis, row_labels = bernstein_linear_matrix_for_diamond(d)
    m, n = A.nrows(), A.ncols()

    if pivot_index is None:
        pivot_index = n - 1

    if pivot_sign not in [1, -1]:
        raise ValueError("pivot_sign must be +1 or -1.")

    p = MixedIntegerLinearProgram(maximization=False)
    c = p.new_variable(integer=True)
    Rvar = p.new_variable(real=True, nonnegative=True)

    # Coefficient bounds
    for j in range(n):
        p.add_constraint(c[j] <= coeff_bound)
        p.add_constraint(c[j] >= -coeff_bound)

    # Nonzero normalization
    p.add_constraint(c[pivot_index] == pivot_sign)

    # Bernstein constraints
    for i in range(m):
        expr = sum(A[i, j] * c[j] for j in range(n))
        p.add_constraint(expr <= Rvar[0])
        p.add_constraint(expr >= -Rvar[0])

    p.set_objective(Rvar[0])

    try:
        opt = p.solve()
    except Exception as e:
        if verbose:
            print("MILP failed:", e)
        return None, None

    cvals = p.get_values(c)
    coeffs = [ZZ(round(cvals[j])) for j in range(n)]

    P = polynomial_from_coeff_vector(coeffs, basis)

    if verbose:
        print("Degree d =", d)
        print("Basis =", basis)
        print("MILP objective R ~", opt)
        print("Candidate P =", P)
        print("Now verifying rigorously by Bernstein bound:")
        verify_catalan_certificate(P, d=d)

    return P, opt

In [8]:
# === MILP with degree-elevated Bernstein constraints ===


def bernstein_linear_matrix_for_diamond_v2(d, bern_deg=None):
    """
    Same as bernstein_linear_matrix_for_diamond, but constraints come from
    Bernstein degree bern_deg >= d. Larger bern_deg gives a tighter LP/MILP relaxation.
    """
    if bern_deg is None:
        bern_deg = 2 * d
    if bern_deg < d:
        raise ValueError("bern_deg must be at least d.")

    basis = monomial_basis_total_degree(d)
    rows = []
    row_labels = []

    monomial_polys = [x**i * y**j for (i, j) in basis]

    for k in range(4):
        transformed_monomials = [rotate_polynomial(M, k) for M in monomial_polys]
        transformed_coeffs = [
            bernstein_coeffs_standard_triangle(Mk, d=bern_deg)
            for Mk in transformed_monomials
        ]

        for a in range(bern_deg + 1):
            for b in range(bern_deg + 1 - a):
                c = bern_deg - a - b
                row = [transformed_coeffs[col][(a, b, c)] for col in range(len(basis))]
                rows.append(row)
                row_labels.append((k, a, b, c))

    A = matrix(QQ, rows)
    return A, basis, row_labels


def search_candidate_by_milp_v2(d, coeff_bound=10, bern_deg=None,
                                pivot_index=None, pivot_sign=1, verbose=True):
    """
    MILP search using degree-elevated Bernstein constraints.
    """
    A, basis, row_labels = bernstein_linear_matrix_for_diamond_v2(d, bern_deg=bern_deg)
    m, n = A.nrows(), A.ncols()

    if pivot_index is None:
        pivot_index = n - 1
    if pivot_sign not in [1, -1]:
        raise ValueError("pivot_sign must be +1 or -1.")

    p = MixedIntegerLinearProgram(maximization=False)
    c = p.new_variable(integer=True)
    Rvar = p.new_variable(real=True, nonnegative=True)

    for j in range(n):
        p.add_constraint(c[j] <= coeff_bound)
        p.add_constraint(c[j] >= -coeff_bound)

    p.add_constraint(c[pivot_index] == pivot_sign)

    for i in range(m):
        expr = sum(A[i, j] * c[j] for j in range(n))
        p.add_constraint(expr <= Rvar[0])
        p.add_constraint(expr >= -Rvar[0])

    p.set_objective(Rvar[0])

    try:
        opt = p.solve()
    except Exception as e:
        if verbose:
            print("MILP failed:", e)
        return None, None

    cvals = p.get_values(c)
    coeffs = [ZZ(round(cvals[j])) for j in range(n)]
    P = polynomial_from_coeff_vector(coeffs, basis)

    if verbose:
        used_deg = bern_deg if bern_deg is not None else 2 * d
        print("Degree d =", d)
        print("Bernstein degree D =", used_deg)
        print("MILP objective R ~", opt)
        print("Candidate P =", P)
        print("Now verifying with tightened bound:")
        verify_catalan_v2(P, d=d, bern_deg=used_deg, subdiv_depth=2)

    return P, opt

In [9]:
P, opt = search_candidate_by_milp(d=2, coeff_bound=5)

Degree d = 2
Basis = [(0, 0), (0, 1), (1, 0), (0, 2), (1, 1), (2, 0)]
MILP objective R ~ 1.0
Candidate P = x^2 + 2*x*y - y^2
Now verifying rigorously by Bernstein bound:
P = x^2 + 2*x*y - y^2
degree d = 2
Bernstein upper bound B = 1
threshold T = (2 e^(3/2))^(-d) in interval form:
0.012446767091965985744835603912515444157924898047105803891907?
B < T ? False
FAIL: Bernstein bound is not small enough.


In [10]:
P, opt = search_candidate_by_milp(d=6, coeff_bound=20)

Degree d = 6
Basis = [(0, 0), (0, 1), (1, 0), (0, 2), (1, 1), (2, 0), (0, 3), (1, 2), (2, 1), (3, 0), (0, 4), (1, 3), (2, 2), (3, 1), (4, 0), (0, 5), (1, 4), (2, 3), (3, 2), (4, 1), (5, 0), (0, 6), (1, 5), (2, 4), (3, 3), (4, 2), (5, 1), (6, 0)]
MILP objective R ~ 0.2666666666666668
Candidate P = x^6 - 2*x^5*y + 3*x^4*y^2 + 2*x^3*y^3 - 5*x^2*y^4 - 2*x*y^5 + y^6 - 2*x^4 + 2*x^3*y - 4*x^2*y^2 + 2*x*y^3 - 2*y^4 + x^2 - x*y + y^2
Now verifying rigorously by Bernstein bound:
P = x^6 - 2*x^5*y + 3*x^4*y^2 + 2*x^3*y^3 - 5*x^2*y^4 - 2*x*y^5 + y^6 - 2*x^4 + 2*x^3*y - 4*x^2*y^2 + 2*x*y^3 - 2*y^4 + x^2 - x*y + y^2
degree d = 6
Bernstein upper bound B = 4/15
threshold T = (2 e^(3/2))^(-d) in interval form:
1.9282781888543679609005732926567785323773880045146726958513?e-6
B < T ? False
FAIL: Bernstein bound is not small enough.


In [11]:
def scan_milp_candidates(d, coeff_bound=10, max_pivots=None):
    A, basis, row_labels = bernstein_linear_matrix_for_diamond(d)
    n = len(basis)

    if max_pivots is None:
        pivot_indices = range(n)
    else:
        pivot_indices = range(min(max_pivots, n))

    candidates = []

    for pivot_index in pivot_indices:
        for sign in [1, -1]:
            print()
            print("Trying pivot:", pivot_index, basis[pivot_index], "sign", sign)

            P, opt = search_candidate_by_milp(
                d=d,
                coeff_bound=coeff_bound,
                pivot_index=pivot_index,
                pivot_sign=sign,
                verbose=False
            )

            if P is not None and P != 0:
                B = bernstein_bound_on_diamond(P, d=d)
                candidates.append((B, P, opt, pivot_index, sign))
                print("B =", B)
                print("P =", P)

    candidates.sort(key=lambda t: t[0])

    print()
    print("Best candidates:")
    for B, P, opt, pivot_index, sign in candidates[:10]:
        print("B =", B, "pivot =", pivot_index, "sign =", sign)
        print("P =", P)
        print()

    return candidates

In [12]:
cands = scan_milp_candidates(d=4, coeff_bound=10)


Trying pivot: 0 (0, 0) sign 1
B = 1
P = -x^4 - 2*x^3*y - 10*x^2*y^2 + 2*x*y^3 - y^4 - x^2 - y^2 + 1

Trying pivot: 0 (0, 0) sign -1
B = 1
P = -4*x^4 - 8*x^2*y^2 - 2*y^4 + 4*x^2 + 4*y^2 - 1

Trying pivot: 1 (0, 1) sign 1
B = 1/2
P = -x^4 - 2*x^2*y^2 - x^2*y - y^3 + x^2 + y

Trying pivot: 1 (0, 1) sign -1
B = 1/2
P = -x^4 - 2*x^2*y^2 + x^2*y + y^3 + x^2 - y

Trying pivot: 2 (1, 0) sign 1
B = 1/2
P = -2*x^2*y^2 - y^4 - x^3 - x*y^2 + y^2 + x

Trying pivot: 2 (1, 0) sign -1
B = 1/2
P = -2*x^2*y^2 - y^4 + x^3 + x*y^2 + y^2 - x

Trying pivot: 3 (0, 2) sign 1
B = 1/2
P = x^4 + x^3*y + x^2*y^2 + x*y^3 - y^4 - x^2 - x*y + y^2

Trying pivot: 3 (0, 2) sign -1
B = 1/2
P = y^4 - x^3 - x*y^2 - y^2 + x

Trying pivot: 4 (1, 1) sign 1
B = 1/3
P = -2*x^3*y + x*y

Trying pivot: 4 (1, 1) sign -1
B = 1/3
P = -x*y

Trying pivot: 5 (2, 0) sign 1
B = 1/2
P = -x^4 - x*y^3 + x*y^2 + x^2

Trying pivot: 5 (2, 0) sign -1
B = 1/2
P = x^4 - x^2*y - y^3 - x^2 + y

Trying pivot: 6 (0, 3) sign 1
B = 1/2
P = -x^3*y + y^

In [13]:
B, P, opt, pivot_index, sign = cands[0]
verify_catalan_certificate(P, d=4)

P = x^2*y^2
degree d = 4
Bernstein upper bound B = 1/6
threshold T = (2 e^(3/2))^(-d) in interval form:
0.00015492201104164740144032296442604174321915497409587156567992?
B < T ? False
FAIL: Bernstein bound is not small enough.


(False, 1/6, 0.00015492201104164740144032296442604174321915497409587156567992?)

In [14]:
P = 1 - 6*x**2 - 6*y**2 - x**2*y**2
verify_catalan_v2(P, bern_deg=8, subdiv_depth=2)


P = -x^2*y^2 - 6*x^2 - 6*y^2 + 1
degree d = 4

--- Bounds comparison ---
  Numerical ||P||_S  >=  5.00000000000000
  Bernstein naive (D=d=4):  5
  Degree elevation (D=8):   5  (5.000000)
  + Subdivision (depth=2, 64 triangles):  5  (5.000000)

  Threshold T = (2e^(3/2))^(-4) ~ 1.549220e-04
  B_best / T ~ 32274.30

FAIL: B_best < T not satisfied.
  Bound / true sup ~ 1.0000 (tightness ratio, 1.0 = perfect)


(False, 5, 0.00015492201104164740144032296442604174321915497409587156567992?)

In [19]:
P, opt = search_candidate_by_milp_v2(d=7, coeff_bound=10000, bern_deg=7)

Degree d = 7
Bernstein degree D = 7
MILP objective R ~ 0.19047619047619044
Candidate P = x^7 - x^6*y - 9*x^5*y^2 - 2*x^4*y^3 - x^3*y^4 - x*y^6 - 3*x^3*y^3 - 2*x*y^5 - 2*x^5 + 2*x^3*y^2 + 2*x^2*y^3 + 2*x*y^4 + x*y^3 + x^3 - x*y^2
Now verifying with tightened bound:
P = x^7 - x^6*y - 9*x^5*y^2 - 2*x^4*y^3 - x^3*y^4 - x*y^6 - 3*x^3*y^3 - 2*x*y^5 - 2*x^5 + 2*x^3*y^2 + 2*x^2*y^3 + 2*x*y^4 + x*y^3 + x^3 - x*y^2
degree d = 7

--- Bounds comparison ---
  Numerical ||P||_S  >=  0.0962712076412700
  Bernstein naive (D=d=7):  4/21
  Degree elevation (D=7):   4/21  (0.190476)
  + Subdivision (depth=2, 64 triangles):  361/3584  (0.100725)

  Threshold T = (2e^(3/2))^(-7) ~ 2.151285e-07
  B_best / T ~ 468210.59

FAIL: B_best < T not satisfied.
  Bound / true sup ~ 1.0463 (tightness ratio, 1.0 = perfect)
